# IIS — core pipeline (your original six steps)

1. Copula transform  2. Subset entropies  3. Möbius inversion  4. Architecture mass
5. Null calibration  6. Normalize

Only two things differ from the original pipeline:
* **Step 2** uses the closed-form Gaussian-copula entropy instead of kNN (removes the estimator error).
* **Step 2** counting works for discrete variables with any number of categories.
* **Step 5** subtracts the average of 100 fresh independent reference datasets instead of one fixed dataset
  (removes the noise floor), and reports $Z_{total}$: how far the data sit above that noise.

Every step is checked against the exact value computed from the generating distribution.

In [8]:
import itertools
import numpy as np
import pandas as pd
from scipy.stats import norm, rankdata

N, n, RHO, B = 5, 1000, 0.6, 100
rng = np.random.default_rng(2026)
SUBSETS = [S for k in range(1, N + 1) for S in itertools.combinations(range(N), k)]
ORDERS = list(range(2, N + 1))
pd.set_option("display.precision", 3)

## The six steps

In [9]:
def step1_copula(X, kind):
    """Continuous: ranks -> normal scores. Discrete: unchanged."""
    if kind == "continuous":
        return norm.ppf(rankdata(X, axis=0) / (len(X) + 1))
    return X

def step2_entropies(Z, kind):
    """Entropy (bits) of every non-empty group of variables."""
    H = {}
    if kind == "continuous":
        C = np.corrcoef(Z, rowvar=False)
        for S in SUBSETS:
            H[S] = 0.5 * (len(S) * np.log2(2 * np.pi * np.e) + np.linalg.slogdet(C[np.ix_(S, S)])[1] / np.log(2))
    else:
        # any number of categories: relabel each variable 0..K-1, then give each joint pattern one integer code
        L = np.column_stack([np.unique(c, return_inverse=True)[1] for c in Z.T])
        K = L.max() + 1
        for S in SUBSETS:
            p = np.bincount(L[:, S] @ (K ** np.arange(len(S)))) / len(Z)
            p = p[p > 0]
            H[S] = -(p * np.log2(p)).sum()
    return H

def step3_mobius(H):
    """mu(S) = sum over subsets T of S of (-1)^(|S|-|T|) H(T)."""
    return {S: sum((-1) ** (len(S) - len(T)) * H[T]
                   for j in range(1, len(S) + 1) for T in itertools.combinations(S, j)) for S in H}

def step4_mass(mu):
    """Architecture mass A_k = sum of |mu(S)| over groups of size k."""
    return np.array([sum(abs(mu[S]) for S in mu if len(S) == k) for k in ORDERS])

def mass(X, kind):
    return step4_mass(step3_mobius(step2_entropies(step1_copula(X, kind), kind)))

def step5_null(X, kind, A):
    """Subtract the average mass of B independent reference datasets; report Z_total."""
    def reference():
        if kind == "continuous":
            return rng.standard_normal(X.shape)                      # independent, same size
        cols = []
        for c in X.T:                                                # independent, same category frequencies
            values, counts = np.unique(c, return_counts=True)
            cols.append(rng.choice(values, size=len(c), p=counts / len(c)))
        return np.column_stack(cols)
    null = np.array([mass(reference(), kind) for _ in range(B)])
    A_star = np.maximum(0, A - null.mean(0))
    Z_total = (A.sum() - null.sum(1).mean()) / null.sum(1).std(ddof=1)
    return A_star, Z_total, null.mean(0)

def step6_normalize(A_star):
    return A_star / A_star.sum() if A_star.sum() > 0 else np.full(len(A_star), np.nan)

def iis(X, kind):
    A = mass(X, kind)
    A_star, Z_total, noise = step5_null(X, kind, A)
    return {"A": A, "noise": noise, "A_star": A_star, "a": step6_normalize(A_star), "Z": Z_total}

## Benchmarks and their exact values

In [10]:
STATES = np.array(list(itertools.product([0, 1], repeat=N)))
AR1 = RHO ** np.abs(np.subtract.outer(range(N), range(N)))
KIND = {"Gaussian IID": "continuous", "Correlated Gaussian": "continuous",
        "Shared Ancestry": "discrete", "XOR Epistasis": "discrete"}

def sample(m):
    if m == "Gaussian IID":
        return rng.standard_normal((n, N))
    if m == "Correlated Gaussian":
        return rng.multivariate_normal(np.zeros(N), AR1, size=n)
    if m == "Shared Ancestry":
        p = np.where(rng.random(n) < 0.5, 0.8, 0.2)[:, None]
        return (rng.random((n, N)) < p).astype(int)
    X = rng.integers(0, 2, (n, N)); X[:, -1] = X[:, :-1].sum(1) % 2
    return X

def exact_H(m):
    if KIND[m] == "continuous":
        C = np.eye(N) if m == "Gaussian IID" else AR1
        return {S: 0.5 * (len(S) * np.log2(2 * np.pi * np.e) + np.linalg.slogdet(C[np.ix_(S, S)])[1] / np.log(2))
                for S in SUBSETS}
    if m == "Shared Ancestry":
        p = sum(0.5 * np.prod(np.where(STATES == 1, q, 1 - q), axis=1) for q in (0.8, 0.2))
    else:
        ok = STATES[:, :-1].sum(1) % 2 == STATES[:, -1]; p = ok / ok.sum()
    H = {}
    for S in SUBSETS:
        q = np.bincount(STATES[:, S] @ (2 ** np.arange(len(S))), weights=p); q = q[q > 0]
        H[S] = -(q * np.log2(q)).sum()
    return H

EXACT_A = {m: step4_mass(step3_mobius(exact_H(m))) for m in KIND}

## Run: 100 datasets per benchmark

In [11]:
runs = {m: [iis(sample(m), KIND[m]) for _ in range(100)] for m in KIND}

## Step 2 check: estimated vs exact subset entropies (bits)
Error of $\hat H(S)$ against the exact $H(S)$ of the generating distribution, pooled over all groups of size $|S|$ and 100 datasets.
Continuous: $\hat H(S)=\tfrac12\log_2[(2\pi e)^{|S|}\det C_S]$ with $C$ the correlation of the normal scores. Discrete: plug-in $-\sum_x \hat p(x)\log_2\hat p(x)$.

In [ ]:
rows = []
for m in KIND:
    Hx = exact_H(m)
    err = {k: [] for k in range(1, N + 1)}
    for _ in range(100):
        Hh = step2_entropies(step1_copula(sample(m), KIND[m]), KIND[m])
        for S in SUBSETS:
            err[len(S)].append(Hh[S] - Hx[S])
    for k, e in err.items():
        e = np.array(e)
        rows.append({"benchmark": m, "|S|": k, "mean error": e.mean(), "sd": e.std(ddof=1), "max |error|": np.abs(e).max()})
entropy_table = pd.DataFrame(rows).set_index(["benchmark", "|S|"])
entropy_table.to_csv("table_step2_entropies.csv")
entropy_table

## Architecture mass at each order: estimate vs exact (bits)
* **A (raw)** — step 4 on the data.
* **Noise** — average mass of the independent reference datasets (step 5).
* **A\*** — raw minus noise. This is the accurate coefficient for each order.
* **Exact** — computed from the generating distribution.

In [12]:
rows = []
for m in KIND:
    R = runs[m]
    for i, k in enumerate(ORDERS):
        rows.append({"benchmark": m, "order k": k,
                     "A (raw)": np.mean([r["A"][i] for r in R]),
                     "noise": np.mean([r["noise"][i] for r in R]),
                     "A* (calibrated)": np.mean([r["A_star"][i] for r in R]),
                     "exact A": EXACT_A[m][i]})
mass_table = pd.DataFrame(rows).set_index(["benchmark", "order k"])
mass_table["error"] = mass_table["A* (calibrated)"] - mass_table["exact A"]
mass_table.to_csv("table_architecture_mass.csv")
mass_table

A (raw)      noise  A* (calibrated)    exact A  \
benchmark           order k                                                     
Gaussian IID        2        7.113e-03  7.263e-03        1.435e-03  0.000e+00   
                    3        2.324e-04  2.338e-04        7.316e-05  8.882e-15   
                    4        6.582e-06  6.116e-06        2.900e-06  8.882e-15   
                    5        8.484e-08  7.470e-08        4.832e-08  7.105e-15   
Correlated Gaussian 2        1.669e+00  7.243e-03        1.662e+00  1.669e+00   
                    3        4.770e-01  2.324e-04        4.768e-01  4.749e-01   
                    4        1.077e-01  6.060e-06        1.077e-01  1.056e-01   
                    5        1.300e-02  7.479e-08        1.300e-02  1.222e-02   
Shared Ancestry     2        9.509e-01  7.192e-03        9.437e-01  9.562e-01   
                    3        4.490e-01  7.281e-03        4.417e-01  4.648e-01   
                    4        1.701e-01  3.753e-03        1.663e-01  1.684e-01   
                    5        3.248e-02  8.075e-04        3.167e-02  3.324e-02   
XOR Epistasis       2        7.998e-03  7.192e-03        1.822e-03  0.000e+00   
                    3        8.070e-03  7.258e-03        1.809e-03  0.000e+00   
                    4        3.722e-03  3.739e-03        8.192e-04  0.000e+00   
                    5        9.884e-01  7.916e-04        9.876e-01  1.000e+00   

                                 error  
benchmark           order k             
Gaussian IID        2        1.435e-03  
                    3        7.316e-05  
                    4        2.900e-06  
                    5        4.832e-08  
Correlated Gaussian 2       -7.547e-03  
                    3        1.915e-03  
                    4        2.113e-03  
                    5        7.857e-04  
Shared Ancestry     2       -1.251e-02  
                    3       -2.310e-02  
                    4       -2.121e-03  
                    5       -1.570e-03  
XOR Epistasis       2        1.822e-03  
                    3        1.809e-03  
                    4        8.192e-04  
                    5       -1.242e-02

## IIS structure and Z_total
The structure is read only when $Z_{total} > 3$.

In [13]:
rows = []
for m in KIND:
    R = runs[m]; Z = np.array([r["Z"] for r in R]); keep = [r for r in R if r["Z"] > 3]
    ex = step6_normalize(EXACT_A[m]) if EXACT_A[m].sum() > 1e-9 else np.full(4, np.nan)
    row = {"benchmark": m, "median Z_total": np.median(Z), "share Z_total > 3": np.mean(Z > 3)}
    for i, k in enumerate(ORDERS):
        if len(keep) >= 50:
            v = np.array([r["a"][i] for r in keep])
            row[f"a{k}"] = f"{v.mean():.3f} [{np.quantile(v, .025):.3f}, {np.quantile(v, .975):.3f}]"
        else:
            row[f"a{k}"] = "below noise"
        row[f"exact a{k}"] = ex[i]
    rows.append(row)
structure_table = pd.DataFrame(rows).set_index("benchmark")
structure_table.to_csv("table_iis_structure_core.csv")
structure_table

,median Z_total,share Z_total > 3,a2,exact a2,a3,exact a3,a4,exact a4,a5,exact a5
benchmark,,,,,,,,,,
Gaussian IID,-0.353,0.02,below noise,NaN,below noise,NaN,below noise,NaN,below noise,NaN
Correlated Gaussian,669.081,1.00,"0.738 [0.695, 0.806]",0.738,"0.210 [0.170, 0.233]",0.210,"0.047 [0.023, 0.065]",0.047,"0.006 [0.002, 0.009]",0.005
Shared Ancestry,298.687,1.00,"0.597 [0.576, 0.618]",0.589,"0.279 [0.264, 0.291]",0.286,"0.105 [0.090, 0.122]",0.104,"0.020 [0.013, 0.029]",0.020
XOR Epistasis,185.068,1.00,"0.002 [0.000, 0.009]",0.000,"0.002 [0.000, 0.010]",0.000,"0.001 [0.000, 0.004]",0.000,"0.996 [0.981, 1.000]",1.000
